In [1]:
import os
import pandas as pd
from functools import reduce

# === Step 1: 扫描并加载所有模型的 CSV 文件 ===
def load_all_model_results(base_dir):
    csv_paths = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_paths.append(os.path.join(root, file))
    return csv_paths

# === Step 2: 提取每个模型的 F1-macro 并记录其数据集 ===
def extract_f1_macro(csv_paths):
    f1_data = []
    dataset_per_model = {}

    for path in csv_paths:
        model_name = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        if 'Dataset' in df.columns and 'F1-macro' in df.columns:
            df_subset = df[['Dataset', 'F1-macro']].copy()
            df_subset['Model'] = model_name
            f1_data.append(df_subset)
            dataset_per_model[model_name] = set(df_subset['Dataset'].unique())

    f1_all = pd.concat(f1_data, ignore_index=True)
    return f1_all, dataset_per_model

# === Step 3: 仅保留所有模型都跑过的数据集 ===
def get_common_datasets(dataset_per_model):
    return reduce(set.intersection, dataset_per_model.values())

# === Step 4: 构建比较表并计算平均 F1-macro ===
def build_comparison_table(f1_all, common_datasets):
    f1_common = f1_all[f1_all['Dataset'].isin(common_datasets)]
    return f1_common.groupby(['Dataset', 'Model'])['F1-macro'].mean().unstack()

# === Step 5: 生成 LaTeX 表格并加粗每行最优值 ===
def generate_latex_table_with_bold(df, caption="F1-macro Comparison", label="tab:f1_macro_comparison"):
    latex = []
    latex.append(r"\begin{table}[htbp]")
    latex.append(r"\centering")
    latex.append(r"\caption{" + caption + r"}")
    latex.append(r"\label{" + label + r"}")
    latex.append(r"\begin{tabular}{" + "l" + "c" * len(df.columns) + "}")
    latex.append(r"\hline")

    header = ["Dataset"] + list(df.columns)
    latex.append(" & ".join([r"\textbf{" + h + "}" for h in header]) + r" \\")
    latex.append(r"\hline")

    for idx, row in df.iterrows():
        max_val = row.max()
        formatted = [f"\\textbf{{{v:.3f}}}" if v == max_val else f"{v:.3f}" for v in row]
        latex.append(f"{idx} & " + " & ".join(formatted) + r" \\")

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{table}")
    return "\n".join(latex)

# === Step 6: 主流程入口 ===
def run_full_comparison(base_dir):
    csv_paths = load_all_model_results(base_dir)
    f1_all, dataset_per_model = extract_f1_macro(csv_paths)
    common_datasets = get_common_datasets(dataset_per_model)
    if not common_datasets:
        raise ValueError("No common datasets found across all models.")
    f1_table = build_comparison_table(f1_all, common_datasets)
    latex_code = generate_latex_table_with_bold(f1_table)
    return f1_table, latex_code

# === Example Usage ===
base_dir = "./"
f1_df, latex_str = run_full_comparison(base_dir)
print(latex_str)


\begin{table}[htbp]
\centering
\caption{F1-macro Comparison}
\label{tab:f1_macro_comparison}
\begin{tabular}{lcccccccccc}
\hline
\textbf{Dataset} & \textbf{BalanceCascadeClassifier} & \textbf{EasyEnsembleClassifier} & \textbf{OverBaggingClassifier} & \textbf{OverBoostClassifier} & \textbf{RUSBoostClassifier} & \textbf{SMOTEBaggingClassifier} & \textbf{SMOTEBoostClassifier} & \textbf{SelfPacedEnsembleClassifier} & \textbf{UnderBaggingClassifier} & \textbf{results} \\
\hline
x10data.npz & 0.737 & 0.692 & 0.704 & 0.722 & 0.628 & 0.710 & 0.709 & 0.726 & 0.696 & \textbf{0.746} \\
x11data.npz & \textbf{0.528} & 0.487 & 0.481 & 0.503 & 0.474 & 0.481 & 0.511 & 0.489 & 0.525 & 0.481 \\
x12data.npz & \textbf{0.617} & 0.579 & 0.540 & 0.577 & 0.540 & 0.549 & 0.576 & 0.602 & 0.594 & 0.600 \\
x13data.npz & \textbf{0.900} & 0.851 & 0.851 & 0.861 & 0.785 & 0.876 & 0.878 & 0.881 & 0.822 & 0.869 \\
x14data.npz & 0.933 & 0.872 & 0.935 & 0.887 & 0.750 & 0.931 & 0.872 & \textbf{0.940} & 0.895 & 0.933 \\
x1

In [3]:
import os
import pandas as pd
from functools import reduce

# Step 1: 扫描并加载所有模型的 CSV 文件
def load_all_model_results(base_dir):
    csv_paths = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_paths.append(os.path.join(root, file))
    return csv_paths

# Step 2: 提取每个模型的 F1-macro 列
def extract_f1_macro(csv_paths):
    f1_data = []
    dataset_per_model = {}

    for path in csv_paths:
        model_name = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        if 'Dataset' in df.columns and 'F1-macro' in df.columns:
            df_subset = df[['Dataset', 'F1-macro']].copy()
            df_subset['Model'] = model_name
            f1_data.append(df_subset)
            dataset_per_model[model_name] = set(df_subset['Dataset'].unique())

    f1_all = pd.concat(f1_data, ignore_index=True)
    return f1_all, dataset_per_model

# Step 3: 获取所有模型都覆盖的数据集
def get_common_datasets(dataset_per_model):
    return reduce(set.intersection, dataset_per_model.values())

# Step 4: 构建公共数据集上的平均 F1 表格
def build_comparison_table(f1_all, common_datasets):
    f1_common = f1_all[f1_all['Dataset'].isin(common_datasets)]
    return f1_common.groupby(['Dataset', 'Model'])['F1-macro'].mean().unstack()

# Step 5: 生成 LaTeX 表格，最优值加粗+灰色背景
def generate_latex_table_with_gray_highlight(df, caption="F1-macro Comparison", label="tab:f1_macro_comparison"):
    model_short_names = {
        "BalanceCascadeClassifier": "BC",
        "EasyEnsembleClassifier": "EE",
        "OverBaggingClassifier": "OBa",
        "OverBoostClassifier": "OB",
        "RUSBoostClassifier": "RUSB",
        "SMOTEBaggingClassifier": "SBa",
        "SMOTEBoostClassifier": "SB",
        "SelfPacedEnsembleClassifier": "SPE",
        "UnderBaggingClassifier": "UB",
        "results": "ours"
    }

    df_renamed = df.rename(columns=lambda x: model_short_names.get(x, x))

    latex = []
    latex.append(r"\documentclass{article}")
    latex.append(r"\usepackage[table,xcdraw]{xcolor}")
    latex.append(r"\usepackage{adjustbox}")
    latex.append(r"\usepackage[margin=1in]{geometry}")
    latex.append(r"\definecolor{graybg}{gray}{0.9}")
    latex.append(r"\begin{document}")
    latex.append("")
    latex.append(r"\begin{table}[htbp]")
    latex.append(r"\centering")
    latex.append(r"\caption{" + caption + r"}")
    latex.append(r"\begin{adjustbox}{width=\textwidth}")
    latex.append(r"\label{" + label + r"}")
    latex.append(r"\begin{tabular}{" + "l" + "c" * len(df_renamed.columns) + "}")
    latex.append(r"\hline")

    header = ["\\textbf{Dataset}"] + [f"\\textbf{{{col}}}" for col in df_renamed.columns]
    latex.append(" & ".join(header) + r" \\")
    latex.append(r"\hline")

    for idx, row in df_renamed.iterrows():
        max_val = row.max()
        formatted = []
        for val in row:
            if val == max_val:
                formatted.append(r"\cellcolor{graybg}\textbf{" + f"{val:.3f}" + "}")
            else:
                formatted.append(f"{val:.3f}")
        latex.append(f"{idx} & " + " & ".join(formatted) + r" \\")

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{adjustbox}")
    latex.append(r"\end{table}")
    latex.append("")
    latex.append(r"\end{document}")

    return "\n".join(latex)

# Step 6: 主流程入口
def run_full_comparison(base_dir):
    csv_paths = load_all_model_results(base_dir)
    f1_all, dataset_per_model = extract_f1_macro(csv_paths)
    common_datasets = get_common_datasets(dataset_per_model)
    if not common_datasets:
        raise ValueError("No common datasets found across all models.")
    f1_table = build_comparison_table(f1_all, common_datasets)
    latex_code = generate_latex_table_with_gray_highlight(f1_table)
    return f1_table, latex_code

# === 示例使用 ===
base_dir = "./"
f1_df, latex_str = run_full_comparison(base_dir)
with open("f1_macro_table.tex", "w") as f:
    f.write(latex_str)


In [7]:
import os
import pandas as pd
from functools import reduce

# Step 1: 扫描所有模型的 CSV 文件路径
def load_all_model_results(base_dir):
    csv_paths = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_paths.append(os.path.join(root, file))
    return csv_paths

# Step 2: 提取指定性能指标的数据
def extract_metric(csv_paths, metric="F1-macro"):
    data = []
    dataset_per_model = {}

    for path in csv_paths:
        model_name = os.path.splitext(os.path.basename(path))[0]
        df = pd.read_csv(path)
        if 'Dataset' in df.columns and metric in df.columns:
            df_subset = df[['Dataset', metric]].copy()
            df_subset['Model'] = model_name
            data.append(df_subset)
            dataset_per_model[model_name] = set(df_subset['Dataset'].unique())

    all_data = pd.concat(data, ignore_index=True)
    return all_data, dataset_per_model

# Step 3: 获取所有模型覆盖的公共数据集
def get_common_datasets(dataset_per_model):
    return reduce(set.intersection, dataset_per_model.values())

# Step 4: 构建比较表（按模型列，按数据集行）
def build_comparison_table(all_data, common_datasets, metric):
    filtered = all_data[all_data['Dataset'].isin(common_datasets)]
    return filtered.groupby(['Dataset', 'Model'])[metric].mean().unstack()

# Step 5: LaTeX 表格生成，支持并列最优标记
def generate_latex_table_with_gray_highlight(df, metric="F1-macro", caption=None, label=None):
    if caption is None:
        caption = f"{metric} Comparison"
    if label is None:
        label = f"tab:{metric.lower()}_comparison"

    # 简称映射
    model_short_names = {
        "BalanceCascadeClassifier": "BC",
        "EasyEnsembleClassifier": "EE",
        "OverBaggingClassifier": "OBa",
        "OverBoostClassifier": "OB",
        "RUSBoostClassifier": "RUSB",
        "SMOTEBaggingClassifier": "SBa",
        "SMOTEBoostClassifier": "SB",
        "SelfPacedEnsembleClassifier": "SPE",
        "UnderBaggingClassifier": "UB",
        "results": "ours"
    }
    df_renamed = df.rename(columns=lambda x: model_short_names.get(x, x))

    # 保留三位小数用于比较
    rounded_df = df_renamed.round(3)

    latex = []
    latex.append(r"\documentclass{article}")
    latex.append(r"\usepackage[table,xcdraw]{xcolor}")
    latex.append(r"\usepackage{adjustbox}")
    latex.append(r"\usepackage[margin=1in]{geometry}")
    latex.append(r"\definecolor{graybg}{gray}{0.9}")
    latex.append(r"\begin{document}")
    latex.append("")
    latex.append(r"\begin{table}[htbp]")
    latex.append(r"\centering")
    latex.append(r"\caption{" + caption + r"}")
    latex.append(r"\begin{adjustbox}{width=\textwidth}")
    latex.append(r"\label{" + label + r"}")
    latex.append(r"\begin{tabular}{" + "l" + "c" * len(df_renamed.columns) + "}")
    latex.append(r"\hline")

    header = ["\\textbf{Dataset}"] + [f"\\textbf{{{col}}}" for col in df_renamed.columns]
    latex.append(" & ".join(header) + r" \\")
    latex.append(r"\hline")

    for idx, row in rounded_df.iterrows():
        max_val = row.max()
        formatted = []
        for val in row:
            if round(val, 3) == round(max_val, 3):
                formatted.append(r"\cellcolor{graybg}\textbf{" + f"{val:.3f}" + "}")
            else:
                formatted.append(f"{val:.3f}")
        latex.append(f"{idx} & " + " & ".join(formatted) + r" \\")

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{adjustbox}")
    latex.append(r"\end{table}")
    latex.append("")
    latex.append(r"\end{document}")

    return "\n".join(latex)

# Step 6: 主流程
def run_full_comparison(base_dir, metric="F1-macro"):
    csv_paths = load_all_model_results(base_dir)
    all_data, dataset_per_model = extract_metric(csv_paths, metric=metric)
    common_datasets = get_common_datasets(dataset_per_model)
    if not common_datasets:
        raise ValueError("No common datasets found across all models.")
    comparison_table = build_comparison_table(all_data, common_datasets, metric)
    latex_code = generate_latex_table_with_gray_highlight(comparison_table, metric=metric)
    return comparison_table, latex_code

# === 使用示例 ===
base_dir = "./"
metric = "Gmean"  # 或 "AUC", "AUPR", "Gmean"
table, latex_code = run_full_comparison(base_dir, metric)
with open(f"{metric.lower()}_comparison.tex", "w") as f:
    f.write(latex_code)
